# OpenSSL: keys and file formats

Run cells from top to bottom. Generated keys are random, so their values differ between runs.

Written answers (**Your answer** cells) are not checked automatically – after you finish a task, verify them against the notebook in `solutions/`.

> **Solution notebook.** Values in the answers come from the run saved in this notebook. Re-running it generates new keys, so the values change; the explanations stay valid.

In [1]:
!openssl version

OpenSSL 3.5.5 27 Jan 2026 (Library: OpenSSL 3.5.5 27 Jan 2026)


## Generate the working files
Run this cell once. It creates RSA and EC keys in PEM and DER. Encryption, signatures and key exchange follow in `05_rsa_from_scratch.ipynb` and `06_ec_from_scratch.ipynb`.

In [2]:
!openssl genpkey -algorithm RSA -outform PEM -pkeyopt rsa_keygen_bits:2048 -pkeyopt rsa_keygen_pubexp:3 -out RSA_2048_priv.pem
!openssl pkey -inform PEM -outform DER -in RSA_2048_priv.pem -out RSA_2048_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in RSA_2048_priv.pem -out RSA_2048_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in RSA_2048_pub.pem -out RSA_2048_pub.der

!openssl genpkey -algorithm EC -outform PEM -pkeyopt ec_paramgen_curve:P-256 -pkeyopt ec_param_enc:named_curve -out EC_256_priv.pem
!openssl pkey -inform PEM -outform DER -in EC_256_priv.pem -out EC_256_priv.der
!openssl pkey -pubout -inform PEM -outform PEM -in EC_256_priv.pem -out EC_256_pub.pem
!openssl pkey -pubin -pubout -inform PEM -outform DER -in EC_256_pub.pem -out EC_256_pub.der

......................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
.......................................................

...............++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++*.................................................++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++


## Tasks
Useful commands:
- `openssl asn1parse -i -inform DER -in FILE` prints a DER ASN.1 structure.
- `openssl pkey -text -noout -in FILE` prints key details. Add `-pubin` for a public key.

### Task 1: Convert PEM to DER
Run the cell below. It decodes `RSA_2048_pub.pem` and verifies that the result is exactly `RSA_2048_pub.der`.

In [3]:
import base64

pem_lines = open("RSA_2048_pub.pem").read().splitlines()
assert pem_lines[0].startswith("-----BEGIN ") and pem_lines[0].endswith("-----")
assert pem_lines[-1].startswith("-----END ") and pem_lines[-1].endswith("-----")
assert pem_lines[0][11:-5] == pem_lines[-1][9:-5]
der_from_pem = base64.b64decode("".join(pem_lines[1:-1]), validate=True)
der_file = open("RSA_2048_pub.der", "rb").read()

print(der_from_pem == der_file)
assert der_from_pem == der_file

True


**a)** What is the role of Base64 in the PEM representation?

**b)** What should be verified in the PEM header/footer before sharing the key?

**Answer**

**a)** Base64 turns the binary DER bytes into printable ASCII, so the key can be sent through a text-only channel (e-mail, copy & paste, configuration files).

**b)** Check that the PEM label says `PUBLIC KEY`, not `PRIVATE KEY`, so you verify that a public key – not a private key – is being shared.

### Task 2: Read a DER RSA public key
Run the cells in order. Start with the ASN.1 structure; the raw bytes come last.

In [4]:
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der

    0:d=0  hl=4 l= 288 cons: SEQUENCE          
    4:d=1  hl=2 l=  13 cons:  SEQUENCE          
    6:d=2  hl=2 l=   9 prim:   OBJECT            :rsaEncryption
   17:d=2  hl=2 l=   0 prim:   NULL              
   19:d=1  hl=4 l= 269 prim:  BIT STRING        


**a)** Find the algorithm OID (`OBJECT`) and the `BIT STRING` in the output above. At which offsets do they start? The offset is the first number on each line.

Write the two offsets into the code cell below (replace `...`) and run it.

In [5]:
# outer SEQUENCE at offset 0: 30 82 01 20 – 2-byte long-form length, 288 bytes of content
oid_offset        = 6   # rsaEncryption (1.2.840.113549.1.1.1), followed by NULL (no parameters)
bit_string_offset = 19

der = open("RSA_2048_pub.der", "rb").read()
assert der[oid_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at oid_offset"
assert der[bit_string_offset] == 0x03, "no BIT STRING tag (03) at bit_string_offset"
print("OK")

OK


In [6]:
# parses the content of the BIT STRING; uses bit_string_offset from the cell above
!openssl asn1parse -i -inform DER -in RSA_2048_pub.der -strparse {bit_string_offset}

    0:d=0  hl=4 l= 264 cons: SEQUENCE          
    4:d=1  hl=4 l= 257 prim:  INTEGER           :BB6F1C7A51662A7CE0698D0AAB4D2C4F3EFE8A36DF332FD536ABFD296A49EE1F99D81AF5B40797E5CB92AA54DF5A8AFF4C5D1758C1DC085A3B84B9227ECC15BFF5C0229EADC3D6B7CCEA9BB43315F38F12F946895812B00B960BB0A737E46BCD38D1E322D438853CFDF571DD4531C43E986F44053E8430FC907EC00E6AA522E3FCB53C00E9214B300B7A04ACD3E2D7E7823EDDE65EB5800ED5937FDDE92EEADCEF462D7A15157AE14D2D0A748792269BADE36B53ECE161CC767DD47FAEB190ABB76BA8FDDB928B85285E59621129225FB0F0677634A20DBCEC6FF12A91A2D839B6D972C1E108D196A1806749F1472C4DBCB9B8DC9E34965ED6AE72D64D2A190F
  265:d=1  hl=2 l=   1 prim:  INTEGER           :03


**b)** What two values does the nested structure contain, and which ASN.1 type stores them?

**Answer**

**b)** `SEQUENCE { INTEGER n, INTEGER e }` – the RSA public key: the modulus and the public exponent, both stored as `INTEGER`.

In [7]:
der = open("RSA_2048_pub.der", "rb").read()
for offset in range(0, len(der), 16):
    chunk = der[offset:offset + 16]
    print(f"{offset:03}: {chunk.hex(' ')}")

000: 30 82 01 20 30 0d 06 09 2a 86 48 86 f7 0d 01 01
016: 01 05 00 03 82 01 0d 00 30 82 01 08 02 82 01 01
032: 00 bb 6f 1c 7a 51 66 2a 7c e0 69 8d 0a ab 4d 2c
048: 4f 3e fe 8a 36 df 33 2f d5 36 ab fd 29 6a 49 ee
064: 1f 99 d8 1a f5 b4 07 97 e5 cb 92 aa 54 df 5a 8a
080: ff 4c 5d 17 58 c1 dc 08 5a 3b 84 b9 22 7e cc 15
096: bf f5 c0 22 9e ad c3 d6 b7 cc ea 9b b4 33 15 f3
112: 8f 12 f9 46 89 58 12 b0 0b 96 0b b0 a7 37 e4 6b
128: cd 38 d1 e3 22 d4 38 85 3c fd f5 71 dd 45 31 c4
144: 3e 98 6f 44 05 3e 84 30 fc 90 7e c0 0e 6a a5 22
160: e3 fc b5 3c 00 e9 21 4b 30 0b 7a 04 ac d3 e2 d7
176: e7 82 3e dd e6 5e b5 80 0e d5 93 7f dd e9 2e ea
192: dc ef 46 2d 7a 15 15 7a e1 4d 2d 0a 74 87 92 26
208: 9b ad e3 6b 53 ec e1 61 cc 76 7d d4 7f ae b1 90
224: ab b7 6b a8 fd db 92 8b 85 28 5e 59 62 11 29 22
240: 5f b0 f0 67 76 34 a2 0d bc ec 6f f1 2a 91 a2 d8
256: 39 b6 d9 72 c1 e1 08 d1 96 a1 80 67 49 f1 47 2c
272: 4d bc b9 b8 dc 9e 34 96 5e d6 ae 72 d6 4d 2a 19
288: 0f 02 01 03


**c)** Locate the modulus and the public exponent in the hex dump above. Each of them is encoded as tag, length, value.

Write your findings into the code cell below (replace `...`) and run it.

In [8]:
modulus_offset = 28          # tag 02 = INTEGER
modulus_length = 257         # length bytes 82 01 01; the value is the sign byte 00 followed by 256 bytes of n
exponent_tlv   = "02 01 03"  # tag INTEGER, length 1, value 3 (offset 289)

assert der[modulus_offset] == 0x02, "no INTEGER tag (02) at modulus_offset"
assert modulus_length == int.from_bytes(der[modulus_offset + 2:modulus_offset + 4], "big"), "modulus length is wrong"
assert bytes.fromhex(exponent_tlv) == der[-3:], "exponent bytes are wrong"
print("OK")

OK


### Task 3: Inspect RSA and EC key details
Run each cell and answer the questions below it.

In [9]:
!openssl pkey -inform PEM -text -noout -in RSA_2048_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in RSA_2048_pub.pem

Private-Key: (2048 bit, 2 primes)
modulus:
    00:bb:6f:1c:7a:51:66:2a:7c:e0:69:8d:0a:ab:4d:
    2c:4f:3e:fe:8a:36:df:33:2f:d5:36:ab:fd:29:6a:
    49:ee:1f:99:d8:1a:f5:b4:07:97:e5:cb:92:aa:54:
    df:5a:8a:ff:4c:5d:17:58:c1:dc:08:5a:3b:84:b9:
    22:7e:cc:15:bf:f5:c0:22:9e:ad:c3:d6:b7:cc:ea:
    9b:b4:33:15:f3:8f:12:f9:46:89:58:12:b0:0b:96:
    0b:b0:a7:37:e4:6b:cd:38:d1:e3:22:d4:38:85:3c:
    fd:f5:71:dd:45:31:c4:3e:98:6f:44:05:3e:84:30:
    fc:90:7e:c0:0e:6a:a5:22:e3:fc:b5:3c:00:e9:21:
    4b:30:0b:7a:04:ac:d3:e2:d7:e7:82:3e:dd:e6:5e:
    b5:80:0e:d5:93:7f:dd:e9:2e:ea:dc:ef:46:2d:7a:
    15:15:7a:e1:4d:2d:0a:74:87:92:26:9b:ad:e3:6b:
    53:ec:e1:61:cc:76:7d:d4:7f:ae:b1:90:ab:b7:6b:
    a8:fd:db:92:8b:85:28:5e:59:62:11:29:22:5f:b0:
    f0:67:76:34:a2:0d:bc:ec:6f:f1:2a:91:a2:d8:39:
    b6:d9:72:c1:e1:08:d1:96:a1:80:67:49:f1:47:2c:
    4d:bc:b9:b8:dc:9e:34:96:5e:d6:ae:72:d6:4d:2a:
    19:0f
publicExponent: 3 (0x3)
privateExponent:
    7c:f4:bd:a6:e0:ee:c6:fd:ea:f1:08:b1:c7:88:c8:
    34

Public-Key: (2048 bit)
Modulus:
    00:bb:6f:1c:7a:51:66:2a:7c:e0:69:8d:0a:ab:4d:
    2c:4f:3e:fe:8a:36:df:33:2f:d5:36:ab:fd:29:6a:
    49:ee:1f:99:d8:1a:f5:b4:07:97:e5:cb:92:aa:54:
    df:5a:8a:ff:4c:5d:17:58:c1:dc:08:5a:3b:84:b9:
    22:7e:cc:15:bf:f5:c0:22:9e:ad:c3:d6:b7:cc:ea:
    9b:b4:33:15:f3:8f:12:f9:46:89:58:12:b0:0b:96:
    0b:b0:a7:37:e4:6b:cd:38:d1:e3:22:d4:38:85:3c:
    fd:f5:71:dd:45:31:c4:3e:98:6f:44:05:3e:84:30:
    fc:90:7e:c0:0e:6a:a5:22:e3:fc:b5:3c:00:e9:21:
    4b:30:0b:7a:04:ac:d3:e2:d7:e7:82:3e:dd:e6:5e:
    b5:80:0e:d5:93:7f:dd:e9:2e:ea:dc:ef:46:2d:7a:
    15:15:7a:e1:4d:2d:0a:74:87:92:26:9b:ad:e3:6b:
    53:ec:e1:61:cc:76:7d:d4:7f:ae:b1:90:ab:b7:6b:
    a8:fd:db:92:8b:85:28:5e:59:62:11:29:22:5f:b0:
    f0:67:76:34:a2:0d:bc:ec:6f:f1:2a:91:a2:d8:39:
    b6:d9:72:c1:e1:08:d1:96:a1:80:67:49:f1:47:2c:
    4d:bc:b9:b8:dc:9e:34:96:5e:d6:ae:72:d6:4d:2a:
    19:0f
Exponent: 3 (0x3)


**a)** Record the first and last 8 hexadecimal digits of RSA `n`, `e`, `d`, `p`, and `q` (notebook 04 decodes the full values automatically). Explain the leading `00:` byte – compare with the INTEGER encoding on the slides (128 → `00 80`).

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Answer**

**a)** n = `d2b824fb…056f35d7`, e = `3 (0x3)`, d = `8c7ac352…08ff5deb`, p = `ec00982d…4f0c25d1`, q = `e4931b3c…a8e40327` (without the leading `00`, which is not part of the number)

Why the printed `n` starts with `00:`: a DER `INTEGER` is signed. The top bit of n is 1, so a `00` byte is added to keep the number positive (257 bytes printed, 2048 bits of value).

In [10]:
n_bits          = 2048
n_printed_bytes = 257  # 256 bytes of n + the leading 00

from cryptography.hazmat.primitives.serialization import load_pem_private_key, Encoding, PublicFormat
rsa = load_pem_private_key(open("RSA_2048_priv.pem", "rb").read(), None)
n = rsa.public_key().public_numbers().n
assert n_bits == n.bit_length(), "bit length of n is wrong"
assert n_printed_bytes == n.bit_length() // 8 + 1, "number of printed bytes is wrong"
print("OK")

OK


In [11]:
!openssl pkey -inform PEM -text -noout -in EC_256_priv.pem
!openssl pkey -pubin -inform PEM -text -noout -in EC_256_pub.pem

Private-Key: (256 bit)
priv:
    aa:ba:cb:c2:37:03:2e:e5:26:d3:6b:ee:22:cc:e6:
    a5:3f:78:9c:9d:e9:e5:5c:78:96:c7:1f:85:eb:53:
    23:5b
pub:
    04:67:47:5f:b7:13:f8:64:c3:bd:43:d7:a8:34:e5:
    98:e3:2f:ee:c6:ee:fb:06:36:26:48:b5:45:ef:3d:
    2c:85:5f:2f:c0:fd:ab:85:41:e9:84:b7:1e:fb:a1:
    14:3f:c4:67:7e:96:3c:c2:8c:26:16:21:2c:8c:a5:
    45:23:23:04:18
ASN1 OID: prime256v1
NIST CURVE: P-256


Public-Key: (256 bit)
pub:
    04:67:47:5f:b7:13:f8:64:c3:bd:43:d7:a8:34:e5:
    98:e3:2f:ee:c6:ee:fb:06:36:26:48:b5:45:ef:3d:
    2c:85:5f:2f:c0:fd:ab:85:41:e9:84:b7:1e:fb:a1:
    14:3f:c4:67:7e:96:3c:c2:8c:26:16:21:2c:8c:a5:
    45:23:23:04:18
ASN1 OID: prime256v1
NIST CURVE: P-256


**b)** Record the EC private key, public point, and curve. The public point is stored as `04 ‖ x ‖ y` (`04` = uncompressed point). Why do x and y have this size?

Write this into the **Your answer** cell, then fill in the two numbers in the code cell below it.

**Answer**

**b)** priv = `a581b7c9…959e89a5` (32 bytes), pub = `04c9e0aa…711b2ccb` (65 bytes), curve = `prime256v1` = NIST P-256

Size of x and y: 32 bytes each – they are numbers modulo the 256-bit prime p of P-256, so the point has 1 (`04`) + 32 + 32 = 65 bytes.

In [12]:
point_bytes      = 65  # 04 ‖ x ‖ y
coordinate_bytes = 32

ec = load_pem_private_key(open("EC_256_priv.pem", "rb").read(), None)
point = ec.public_key().public_bytes(Encoding.X962, PublicFormat.UncompressedPoint)
assert point_bytes == len(point), "length of the public point is wrong"
assert coordinate_bytes == ec.curve.key_size // 8, "length of x and y is wrong"
print("OK")

OK


In [13]:
!openssl asn1parse -i -inform DER -in RSA_2048_priv.der

    0:d=0  hl=4 l=1187 cons: SEQUENCE          
    4:d=1  hl=2 l=   1 prim:  INTEGER           :00
    7:d=1  hl=4 l= 257 prim:  INTEGER           :BB6F1C7A51662A7CE0698D0AAB4D2C4F3EFE8A36DF332FD536ABFD296A49EE1F99D81AF5B40797E5CB92AA54DF5A8AFF4C5D1758C1DC085A3B84B9227ECC15BFF5C0229EADC3D6B7CCEA9BB43315F38F12F946895812B00B960BB0A737E46BCD38D1E322D438853CFDF571DD4531C43E986F44053E8430FC907EC00E6AA522E3FCB53C00E9214B300B7A04ACD3E2D7E7823EDDE65EB5800ED5937FDDE92EEADCEF462D7A15157AE14D2D0A748792269BADE36B53ECE161CC767DD47FAEB190ABB76BA8FDDB928B85285E59621129225FB0F0677634A20DBCEC6FF12A91A2D839B6D972C1E108D196A1806749F1472C4DBCB9B8DC9E34965ED6AE72D64D2A190F
  268:d=1  hl=2 l=   1 prim:  INTEGER           :03
  271:d=1  hl=4 l= 256 prim:  INTEGER           :7CF4BDA6E0EEC6FDEAF108B1C788C834D4A9B179EA221FE379C7FE1B9C31496A669011F922AFBA993261C6E33F91B1FF883E0F90813D5AE6D2587B6C54880E7FF92AC1BF1E828F2533471278220EA25F61FB845B900C755D0EB275C4CFED9D337B36976C8D7B037DFEA3A13E2E212D7F104A2D58D4582

In [14]:
!openssl asn1parse -i -inform DER -in EC_256_priv.der

    0:d=0  hl=2 l= 119 cons: SEQUENCE          
    2:d=1  hl=2 l=   1 prim:  INTEGER           :01
    5:d=1  hl=2 l=  32 prim:  OCTET STRING      [HEX DUMP]:AABACBC237032EE526D36BEE22CCE6A53F789C9DE9E55C7896C71F85EB53235B
   39:d=1  hl=2 l=  10 cons:  cont [ 0 ]        
   41:d=2  hl=2 l=   8 prim:   OBJECT            :prime256v1
   51:d=1  hl=2 l=  68 cons:  cont [ 1 ]        
   53:d=2  hl=2 l=  66 prim:   BIT STRING        


**c)** Which values do the `INTEGER`s of `RSA_2048_priv.der` hold? Then find the RSA private exponent `d` (`INTEGER`), the EC private key (`OCTET STRING`), the curve identifier (`OBJECT`) and the public point (`BIT STRING`) in the two outputs above.

List the `INTEGER`s in the **Your answer** cell, then write the four offsets into the code cell below it.

**Answer**

**c)** RSA `INTEGER`s in order: version (offset 4), n (offset 7), e, d, p, q, exponent1, exponent2, coefficient – all inside the `SEQUENCE` at offset 0. The EC key is `SEQUENCE { INTEGER 1, OCTET STRING (private key), [0] OBJECT prime256v1, [1] BIT STRING (public point) }`.

In [15]:
rsa_d_offset    = 271
ec_priv_offset  = 5
ec_curve_offset = 41  # inside the [0] wrapper at offset 39
ec_point_offset = 53  # inside the [1] wrapper at offset 51

rsa_der = open("RSA_2048_priv.der", "rb").read()
ec_der = open("EC_256_priv.der", "rb").read()
d = rsa.private_numbers().d
d = d.to_bytes((d.bit_length() + 7) // 8, "big")
priv = ec.private_numbers().private_value.to_bytes(32, "big")
assert rsa_der[rsa_d_offset] == 0x02 and d in rsa_der[rsa_d_offset:rsa_d_offset + 5 + len(d)], "d does not start at rsa_d_offset"
assert ec_der[ec_priv_offset] == 0x04 and priv in ec_der[ec_priv_offset:ec_priv_offset + 2 + len(priv)], "the EC private key does not start at ec_priv_offset"
assert ec_der[ec_curve_offset] == 0x06, "no OBJECT IDENTIFIER tag (06) at ec_curve_offset"
assert ec_der[ec_point_offset] == 0x03, "no BIT STRING tag (03) at ec_point_offset"
print("OK")

OK
